## Machine Learning Model Training
Before diving into the machine learning model training, we need assess the objective of the model. Intrisically, the GDELT data is noisy: noisy means that the data contains little signal about the target variable. This because events are reported differentely by different sources, which have their own biases and agendas. For example, a news source that is biased towards a certain country may report more events that are favorable to that country, while a news source that is biased against that country may report more events that are unfavorable to that country. Or, the same events is treated differently by different sources, which may have different interpretations and positions on the event. For example, a right wing news source may report the USA tariffs of 2025 with a positive tone, while a left wing news source may report the same event with a negative tone. Another aspect to take note of is the intrinsic nature of the time series data we are working with: we can´t predict the future based on the past, because the past is not a good predictor of the future. For example, a country that has experienced a lot of tension in the past may not experience any tension in the future, while a country that has experienced no tension in the past may experience a lot of tension in the future. This is because the world is constantly changing and evolving, and events are influenced by a multitude of factors that are not captured by the GDELT data.
Given this, we think that the average tone of newspapers together with the nature of the events that are reported in a country are good indicators of the future escalation of tension in that country, even though they are not perfect predictors. The average tone of newspapers is a good indicator of the sentiment of the population towards the government and the political situation in the country, while the nature of the events that are reported in a country is a good indicator of the level of tension and conflict in that country.


### Model - XGBoost


Historically, XGBoost has had very good results in time series forecasting problems. It is a gradient boosting algorithms, meaning that it builds an ensemble of weak learners (decision trees) in a sequential manner, where each new tree is trained to correct the errors made by the previous trees. This allows XGBoost to capture complex relationships between the features and the target variable, and to handle non-linearities and interactions between the features. It handles missing values and outliers well. 
Technically, we are gonna train the model using the service `Dataproc Serverless`. Dataproc serverless is a fully managed service that allows us to run Apache Spark and Apache Hadoop jobs without having to manage the underlying infrastructure, meaning that we can focus on writing the code for the model training, sending it to Dataproc serverless and let it handle the scaling, provisioning, and management of the resources needed to run the job, and let it handle the deletion of the resources once the job is completed, letting us to "pay for what we use". 
We are going to use the `pyspark` 

In [ ]:
%%capture
from google.cloud import storage
import os
from dotenv import load_dotenv
from datetime import datetime
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import pandas as pd


load_dotenv()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BIG_QUERY_DATASET")
STAGING_TABLE_NAME = os.getenv("STAGING_TABLE_NAME")
WH_TABLE_NAME = os.getenv("WH_TABLE_NAME")
DM_TABLE_NAME = os.getenv("DM_TABLE_NAME")
BUCKET_NAME = os.getenv("GCP_BUCKET_NAME")

STAGING_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{STAGING_TABLE_NAME}"
WH_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{WH_TABLE_NAME}"
DM_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{DM_TABLE_NAME}"

In [ ]:
%%writefile ml_boosting.py

import sys
import os
import datetime
import json

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from xgboost.spark import SparkXGBClassifier
from google.cloud import storage

def get_session():
    spark = (
        SparkSession.builder
        .appName("ML_job_gdelt_data")
        .getOrCreate()
    )
    return spark

def read_clean_data(input_data_table_id, spark):
    df = spark.read.format("bigquery").option("table", input_data_table_id).load()

    df_clean = df.dropna(
        subset=[
            "avg_tone_ma_5w",
            "goldstein_ma_5w",
            "Target",
            "goldstein_sd_5w",
            "avg_tone_sd_5w",
        ]
    )
    return df_clean

def prepare_split_data(df, cutoff_date="2025-01-01"):
    feature_cols = [
        col
        for col in df.columns
        if col not in ["Target", "Country_code", "week"]
    ]
    
    assembler = VectorAssembler(
        inputCols=feature_cols, outputCol="features", handleInvalid="keep"
    )
    df_transformed = assembler.transform(df)
    
    train_df = df_transformed.filter(df_transformed["week"] < cutoff_date).cache()
    test_df = df_transformed.filter(df_transformed["week"] >= cutoff_date).cache()

    train_df.count()
    test_df.count()
    
    return train_df, test_df, feature_cols

def train_model_XGB(train_df, workers=4):
    classifier = SparkXGBClassifier(
        label_col="Target",
        num_workers=workers,
        eval_metric="logloss",
        tree_method="hist",
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8
    )

    model = classifier.fit(train_df)

    return model
    
def evaluate_model(model, test_df):
    predictions = model.transform(test_df).cache()

    evaluator = BinaryClassificationEvaluator(
        labelCol="Target",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
    cm_df = (
        predictions
        .groupBy("Target", "prediction")
        .count()
        .collect()
    )
    
    tp = fp = tn = fn = 0
    
    for row in cm_df:
        actual = int(row["Target"])
        pred = int(row["prediction"])
        count = int(row["count"])
    
        if actual == 1 and pred == 1:
            tp = count
        elif actual == 0 and pred == 1:
            fp = count
        elif actual == 0 and pred == 0:
            tn = count
        elif actual == 1 and pred == 0:
            fn = count

    total = tp + fp + tn + fn
    auc_score = round(float(evaluator.evaluate(predictions)), 4)
    accuracy = round(float((tp + tn) / total), 4) if total > 0 else 0.0
    specificity = round(float(tn / (tn + fp)), 4) if (tn + fp) > 0 else 0.0

    metrics = {
        "auc_roc": auc_score,
        "accuracy": accuracy,
        "specificity": specificity
    }
    
    return metrics

def export_metrics(metrics, bucket_name, blob_path):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    
    blob.upload_from_string(
        data=json.dumps(metrics, indent=4),
        content_type="application/json"
    )

def save_model(bucket_name, model, feature_cols):
    booster = model.get_booster()
    
    booster.feature_names = feature_cols
    booster.feature_types = ["q"] * len(feature_cols)
    
    raw_json_str = booster.save_raw(raw_format="json").decode("utf-8")
    model_dict = json.loads(raw_json_str)
    
    if "learner" in model_dict:
        model_dict["learner"]["feature_names"] = feature_cols
        model_dict["learner"]["feature_types"] = ["q"] * len(feature_cols)
    
    updated_json_bytes = json.dumps(model_dict, indent=2).encode("utf-8")

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob("models/xgboost_model.json")

    blob.upload_from_string(updated_json_bytes, content_type="application/json")

if __name__ == "__main__":
    PROJECT_ID = sys.argv[1]
    DATASET_ID = sys.argv[2]
    DM_TABLE_NAME = sys.argv[3]
    BUCKET_NAME = sys.argv[4].replace("gs://", "").strip("/")

    DM_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{DM_TABLE_NAME}"
    
    spark = get_session()

    try:
        df_clean = read_clean_data(DM_TABLE_ID, spark)
        tr_df, te_df, feature_cols = prepare_split_data(df_clean)
        model = train_model_XGB(tr_df, workers=4)

        metrics_blob_path = "metrics/xgboost_metrics.json"
        export_metrics(evaluate_model(model, te_df), BUCKET_NAME, metrics_blob_path)

        save_model(BUCKET_NAME, model, feature_cols)

    finally:
        spark.stop()

In [ ]:
%%capture

BATCH_ID = f"ml-boosting-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
BLOB_PATH = "models/xgboost_model.json"
REGION = "europe-west1"
!gcloud storage cp ml_boosting.py gs://{BUCKET_NAME}/scripts/ml_boosting.py

!gcloud dataproc batches submit pyspark gs://{BUCKET_NAME}/scripts/ml_boosting.py \
    --batch={BATCH_ID} \
    --project={PROJECT_ID} \
    --region={REGION} \
    --deps-bucket=gs://{BUCKET_NAME} \
    --properties="spark.dynamicAllocation.enabled=false,spark.dataproc.driverEnv.PIP_PACKAGES=xgboost google-cloud-storage,spark.executorEnv.PIP_PACKAGES=xgboost google-cloud-storage" \
    -- {PROJECT_ID} {DATASET_ID} {DM_TABLE_NAME} {BUCKET_NAME}

In [ ]:
st_client = storage.Client()

blob = st_client.bucket(BUCKET_NAME).blob(BLOB_PATH)

model_bytes = blob.download_as_bytes()

model = xgb.XGBClassifier()
model.load_model(bytearray(model_bytes))

In [ ]:
xgb.plot_importance(model, importance_type='gain',values_format="{v:.0f}")

In [ ]:
df["week"] = pd.to_datetime(df["week"])

df = df.sort_values(["Country_code", "week"])

df["Prediction"] = 1-(
    df.groupby("Country_code")["Target"].shift(1)
)

df_test = df[df["week"] >= pd.Timestamp("2025-01-01")].copy()

df_test = df_test.dropna(subset=["Prediction", "Target"])

auc = roc_auc_score(
    y_true=df_test["Target"],
    y_score=df_test["Prediction"]
)

targets_predictions_df = pd.crosstab(
    df_test["Target"],
    df_test["Prediction"]
)

tp = targets_predictions_df.loc[1, 1]
fp = targets_predictions_df.loc[0, 1]
tn = targets_predictions_df.loc[0, 0]
fn = targets_predictions_df.loc[1, 0]

total = tp + fp + tn + fn

accuracy = round((tp + tn) / total, 4)

specificity = round(tn / (tn + fp), 4)

print(f"Naive Model AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Specificity: {specificity:.4f}")

In [ ]:
!gcloud storage cat gs://{BUCKET_NAME}/metrics/xgboost_metrics.json